<a href="https://colab.research.google.com/github/mohanvidushi50/basic_rag/blob/main/gemapirag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("geminiki")

client = genai.Client(api_key=api_key)

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="who is president of india"
)

print(response.text)

The current President of India is **Droupadi Murmu**.

She assumed office on July 25, 2022, becoming the first person from a tribal community and the second woman to hold the office.


In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = api_key

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai google-genai tiktoken rapidocr-onnxruntime faiss-cpu

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_5696/2557118171.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
with open("vidushi_gawar.txt","r", encoding="utf8") as f:
  data = f.read()

In [ ]:
loder=TextLoader('vidushi_gawar.txt', encoding="utf8")

In [ ]:
document=loder.load()

In [ ]:
print(document[0].page_content)

Dear everyone, today we are here to discuss a very mysterious person: Vidushi. Vidushi is very gadhi. Sometimes she acts like a bhadvi, but in her own mind she believes she is a devi who has everything under control. Honestly, after all this time, I still have no idea what she truly wants in life. One moment she is making plans, the next moment she forgets about them completely. It is a talent that very few people possess. Saachi is also a little gadhi, but at least she is somewhat padi likhi and tries to understand things. However, this story is not about Saachi. Today's entire discussion is dedicated to Vidushi and her legendary ability to avoid studying.Yesterday, the three of us — me, Saachi, and Vidushi — made a grand plan. We decided that we would finally start studying RAG (Retrieval-Augmented Generation) together. We were all excited, full of motivation, and acting like future AI researchers who would publish groundbreaking papers one day.Now comes the interesting part. I woke 

chunking

In [ ]:
!pip show langchain
!pip show langchain-text-splitters

Name: langchain
Version: 1.3.9
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-text-splitters
Version: 1.1.2
Summary: LangChain text splitting utilities
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core
Required-by: langchain-classic


In [ ]:
!pip show langchain
!pip show langchain-community
!pip show langchain-text-splitters

Name: langchain
Version: 1.3.9
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: 
Name: langchain-text-splitters
Version: 1.1.2
Summary: LangChain text splitting utilities
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core
Required-by: langchain-classic


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

In [ ]:
text_chunks=text_splitter.split_documents(document)

In [ ]:
text_chunks

[Document(metadata={'source': 'vidushi_gawar.txt'}, page_content='Dear everyone, today we are here to discuss a very mysterious person: Vidushi. Vidushi is very gadhi. Sometimes she acts like a bhadvi, but in her own mind she believes she is a devi who has everything under control. Honestly, after all this time, I still have no idea what she truly wants in life. One moment she is making plans, the next moment she forgets about them completely. It is a talent that very few people possess. Saachi is also a little gadhi, but at least she is somewhat padi likhi'),
 Document(metadata={'source': 'vidushi_gawar.txt'}, page_content="gadhi, but at least she is somewhat padi likhi and tries to understand things. However, this story is not about Saachi. Today's entire discussion is dedicated to Vidushi and her legendary ability to avoid studying.Yesterday, the three of us — me, Saachi, and Vidushi — made a grand plan. We decided that we would finally start studying RAG (Retrieval-Augmented Genera

In [ ]:

print(text_chunks[3].page_content)

procrastination level 100. Whenever I try to say something serious, she somehow makes me chup. Every time there is work to do, she becomes an expert at postponing things. The energy used to avoid studying could probably be used to finish studying itself. Our enthusiasm levels were completely different. I was trying to follow the plan and stay disciplined, while Vidushi was busy being Vidushi. At this point, Vidushi has become less of a person and more of an unexpected phenomenon that scientists


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

In [ ]:
!pip install faiss-cpu

In [ ]:
vectorstore = FAISS.from_documents(text_chunks, embeddings)

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
output_parser=StrOutputParser()

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=api_key
)

In [ ]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("how is vidushi")

'Vidushi is described as a very mysterious person who is characterized as "very gadhi" and sometimes acts like a "bhadvi." Despite this, she believes she is a "devi" who has everything under control. She has a legendary ability to avoid studying, demonstrating "procrastination level 100." Vidushi is an expert at postponing tasks and often forgets plans she makes. For instance, she completely avoided studying RAG despite making a grand plan with others. She also has a knack for making others silent when they try to say something serious. The context suggests she has become "less of a person and more of an unexpected phenomenon." It is unclear what she truly wants in life.'

In [ ]:
rag_chain.invoke("who is radha")

'I am sorry, but the provided context does not contain any information about Radha. Therefore, I cannot answer who Radha is based on the given documents.'

In [ ]:
res=input("enter ur question")
rag_chain.invoke(res)

enter ur questionis vidushi good or bad


'Based on the provided context, Vidushi is portrayed in a negative light. She is described as "very gadhi" and sometimes acting like a "bhadvi." The text highlights her "procrastination level 100" and her "legendary ability to avoid studying." She frequently forgets plans and becomes an expert at postponing tasks. The narrator expresses frustration, stating that Vidushi makes them "chup" when trying to be serious and that she is "busy being Vidushi" instead of following plans. Overall, the context suggests that Vidushi is perceived as irresponsible and unhelpful, particularly concerning academic commitments.'